# Quasi-Gaussian Model Analysis

In [ ]:
using Pkg
Pkg.activate("../.")

In [ ]:
using CSV
using DataFrames
using DiffFusion
using Plots
using Printf
using StatsBase

plotlyjs()

In [ ]:
file_name = "swaption_2.csv"
table = CSV.read(file_name, DataFrame);

In [ ]:
function plot_smiles(table, exercise_times, swap_time; show_label = true, show_title = true)
    fig = plot()
    forwards = Float64[]
    atm_vols = Float64[]
    for exercise_time in exercise_times
        table_ = table[
            (table.exercise_time .== exercise_time) .&
            (table.swap_time .== swap_time), :
        ]
        if size(table_) != (1, 21)
            error("No smile found.")
        end
        strikes = table_.forward .+ [ parse(Float64, s) for s in names(table)[9:end] ] .* 1e-4
        volatilities = collect(table_[1, 9:end])
        if show_label
            label = @sprintf("%.0fy-%.0fy", exercise_time, swap_time)
        else
            label = nothing
        end
        plot!(
            fig,
            strikes,
            volatilities,
            label = label,
            marker = :circle,
            markersize = 2,
            linewidth = 1,
            linestyle = :dot,
        )
        push!(forwards, table_.forward[1])
        push!(atm_vols, volatilities[7])
    end
    if show_label
        label = "ATM"
    else
        label = nothing
    end
    plot!(
        fig,
        forwards,
        atm_vols,
        label = label,
        linestyle = :solid,
    )
    xlabel!(fig, "strike")
    ylabel!(fig, "normal volatility")
    if show_title
        title!(fig, @sprintf("%.0fy swap term", swap_time), titlefontsize = 10)
    end
    return fig
end

f1 = plot_smiles(table, [1.0, 2.0, 5.0, 10.0], 1.0, show_label = false)
f2 = plot_smiles(table, [1.0, 2.0, 5.0, 10.0], 2.0, show_label = true)
f3 = plot_smiles(table, [1.0, 2.0, 5.0, 10.0], 5.0, show_label = false)
f4 = plot_smiles(table, [1.0, 2.0, 5.0, 10.0], 10.0, show_label = false)
fig = plot(f1, f2, f3, f4, size=(1200, 800), margin=12Plots.mm,)
fig